# Track 2 — Intelligent Well Archive: full pipeline

config check → build or load the evidence store → route → answer → verify → write predictions → validate → cost summary → review queue.

**Quick start (no key needed).** Run all cells. The 32 published questions are answered from the committed cache `indexes/llm_cache.json`: no network, deterministic, same `predictions.csv`.

**Your own questions or a live run.** Set `OPENROUTER_KEY` (environment variable or a line in `.env`) and, for a different question set, `QUESTIONS_CSV` below. Cached answers are always used first; the model is called only for what is not cached. Without a key, uncached questions get a row marked `NOT EVALUATED` instead of an error.

In [ ]:
# First cell: which API key variable is in use (its name only, never its value).
import os, sys
from pathlib import Path

here = Path.cwd()
sys.path.insert(0, str(here if (here / "archive_qa").is_dir() else here / "submission"))

from archive_qa import llm
print(llm.describe_key())

In [ ]:
import pandas as pd
from archive_qa import answer, config, pipeline, route, store, write
from archive_qa.locate import locate_stats

pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 100)

QUESTIONS_CSV = os.environ.get("NAPE_QUESTIONS_CSV") or None   # None = the organiser's 32 published questions. Or a path to a CSV with an id column and a question column.
OUTPUT_PATH = os.environ.get("NAPE_OUTPUT_PATH") or None     # None = submission/predictions.csv (published questions) or predictions_<file>.csv next to it (any other file).
# Optional subset for smoke runs, e.g. NAPE_QUESTION_IDS="T2-016,T2-001". Default: all questions.
QUESTION_IDS = [q for q in os.environ.get("NAPE_QUESTION_IDS", "").split(",") if q] or None

## 1. Config check

In [ ]:
status = config.check()
print(status)
assert status["DATA_ROOT"], status.get("error")
assert status["xml_reports"] == 32 and status["pdf_reports"] == 32
llm_cfg = llm.load_config()
print("LLM:", llm_cfg.describe())       # names and flags only; the key is never shown

## 2. Evidence store (build, or load if already built)

In [ ]:
records = store.build_store(verbose=False) if not config.RECORDS_PATH.exists() else store.load_store()
reports = route.discover_reports(config.XML_DIR)
print(f"{len(records)} records, {len(reports)} reports")
stats = locate_stats(records, show_unlocated=False)

## 3. Route the questions

In [ ]:
questions = pipeline.read_questions(QUESTIONS_CSV)
default_run = pipeline.is_default_questions(QUESTIONS_CSV)
pred_path = pipeline.resolve_output(QUESTIONS_CSV, OUTPUT_PATH)
print(f"{len(questions)} questions from {QUESTIONS_CSV or 'the published question file'}; output: {pred_path}")
routing = route.route_all(questions, reports)
if default_run:
    route.save_routing_report(routing)
print(routing["mode"].value_counts().to_string(), "| early abstentions:", int(routing.early_abstain.sum()))

## 4. Answer and verify
Two cached Haiku runs per question (the second only feeds a disagreement flag), plus one escalation call to Sonnet only if the verifier fails or the model abstains although a matching evidence line exists. Early abstentions make no LLM call. New answers are saved to `indexes/llm_cache_runtime.json`, never to the committed cache.

In [ ]:
client = llm.LLM.from_env()
answers = answer.answer_all(questions, records, reports, client, ids=QUESTION_IDS)
summary = pd.DataFrame([{"question_id": a.question_id, "status": a.status, "mode": a.mode, "confidence": a.confidence,
                         "verified": a.verified, "calls": a.llm_calls, "early_abstain": a.early_abstain,
                         "cited": len(a.cited_ids)} for a in answers])
print(summary["status"].value_counts().to_string())
print(pipeline.not_evaluated_summary(answers) or "every question was evaluated")
summary

## 5. Write predictions, validate with the organiser's checker

In [ ]:
partial = QUESTION_IDS is not None
order = None if default_run else list(questions["question_id"])
write.write_predictions(answers, records, pred_path, order=order, allow_partial=partial)
if default_run and pred_path == config.PREDICTIONS_PATH:
    write.save_answers_json(answers, records)
report = write.validate_predictions(predictions_path=pred_path, expected_ids=None if default_run else order)
print(pred_path.name, "rows:", len(pd.read_csv(pred_path)))
print("validator:", report["validator"])
print("valid:", report["valid"], "| errors:", report["errors"], "| warnings:", report["warnings"])
pd.read_csv(pred_path)[["question_id", "confidence", "answer", "evidence"]]

## 6. Cost summary

In [ ]:
print(f"live calls this run: {client.live_calls}, cache hits: {client.cache_hits}, spent this run: ${client.spent:.4f}")
if default_run and client.live_calls == 0:
    cost_path = llm.write_cost_summary(client.cache, client)
    pd.read_csv(cost_path).tail(8)

## 7. Human review queue
Every answer whose verifier failed, every low-confidence answer, and every NOT EVALUATED row (early abstentions are excluded: they cite what was checked).

In [ ]:
queue = [a for a in answers if a.needs_review]
print(f"{len(queue)} answer(s) need human review")
pd.DataFrame([{"question_id": a.question_id, "confidence": a.confidence, "status": a.status,
               "why": next((n for n in a.notes if n.startswith("VERIFIER FAILED")), "low confidence")} for a in queue])

## 8. Review pack (optional, writes images from the PDFs; git-ignored)

In [ ]:
BUILD_REVIEW = False
if BUILD_REVIEW:
    from eval import review
    print(review.build_review(records, questions))